In [ ]:
from startorch.ingest.fetch_data import show_shard, get_manifest, get_manifest_data, fetch_shard, extract_compact
from startorch.utils import PROJECT_ROOT

# Fetching work entities from OpenAlex S3 storage
RAW_PATH: Path = PROJECT_ROOT/"data"/"openalex"
COMPACT_PATH: Path = PROJECT_ROOT/"data"/"compact"

manifest_path = RAW_PATH/"works"/"manifest.json"

In [ ]:
# Fixing keywords field prefix remnant.
# Rewrites each existing compact shard in place, stripping the leftover
# 'https://openalex.org/' prefix from keyword IDs (extract_compact() strips it
# correctly going forward; already-extracted shards predate that fix).

import os
import duckdb
from pathlib import Path
from startorch.ingest.fetch_data import get_manifest_data

files = get_manifest_data(manifest_path)

db = duckdb.connect()

for i, file in enumerate(files):
    print(f"Fixing file {i+1} out of {len(files)}: {file.key}", flush=True)

    shard_path = COMPACT_PATH/file.key
    tmp_path = Path(str(shard_path) + ".tmp")

    rel = db.read_parquet(shard_path)

    rel = rel.select("""
        * REPLACE (
            list_transform(keywords, a -> {
                'id': replace(a.id, 'https://openalex.org/', ''),
                'display_name': a.display_name,
                'score': a.score
            }) AS keywords,
        ),
    """)

    # Write to a temp file first -- COPY reads shard_path lazily while writing,
    # so writing straight back to shard_path risks reading a partially
    # overwritten file (undefined behavior, possible corruption on the real corpus).
    db.sql(f"""
        COPY (SELECT * FROM rel)
        TO '{tmp_path}'
        (FORMAT parquet, COMPRESSION zstd)
    """)

    os.replace(tmp_path, shard_path)  # atomic on POSIX
